# Trabajo final · Data Scientist · Vidriera

**Tienda online** · Empresa ficticia. Los datos son sintéticos y **se generan por código en este notebook**: ningún resultado representa una situación real.

## La pregunta oficial

> **¿Qué pedidos hay que frenar o revisar antes de despachar?**

**Decisión que habilita:** Cuáles se bloquean, cuáles se revisan a mano (con capacidad para 40 por día) y cuáles se aprueban.

**Tipo de problema:** Clasificación extrema con etiquetas demoradas.

Las instrucciones completas (qué se entrega, criterios de calidad y defensa) están en el PDF **«Trabajo final · Data Scientist · Instrucciones»**. Este notebook es tu lugar de trabajo: vas de los datos sucios a la recomendación sin salir de acá.

| Sección | Módulo del programa |
| --- | --- |
| 1 | Introducción: definición del problema |
| 2 | Python: limpieza |
| 3 | Estadística |
| 4 | Machine Learning y regresión |
| 5 | Deep Learning |
| 6 | Apache Superset |
| 7 | De modelo a producción |
| 8 | Ciencia de datos aplicada al negocio |
| 9 | Fichas y recomendación |

## 0 · Preparación

Ejecutá esta celda una sola vez. `xgboost` es opcional: si querés usarlo, descomentá la última línea.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib
# %pip install -q xgboost


## 1 · Datos y definición del problema

La celda siguiente **genera las tablas**. Tarda unos segundos. Después mirá qué hay con `inventario()` y `diccionario()`.

Antes de modelar, definí el problema: **tipo de problema, población, período, granularidad y variable objetivo**, y qué información está realmente disponible en el momento de decidir.

In [ ]:
import zlib, base64
TEXTO = (
    "eNqlWd1y2zYWvs9TYNh1SyYyI9mW7dj1zmad6e7ObGNNmp1pVqPhQCQksyEJlj9KnNT7Lr3sRa/6CH6x/Q4AkqBEud1Zjy0RwMEB"
    "cH6+84GO01wWFcvqNL9jvGRZPmI5zyI84jePnjyJxIqtRSYKXgQRr2TpliKNk4RfHY2PTs9GbCOKiEdX3/CkFN7FE4afgl1Bk19A"
    "kUx9aOB1UgVFtm7mepfs9TWETsb4uWTRGM955L+NU1FWPM1dB8qnh+MJfh0Iv8L40dnxJRMfq4KHYSyz4RmT8eFk6nhqFzypeBAO"
    "yZ00mtkzGqxkUGE8EpjgYt8lTOIWfpxVYi2K0h2P2OkJPl5fe96I1VlcXTmvzCJZsOSlwCKT43N1lpUIbznaONOu8r7SVyMz3db6"
    "u7POsZfxzszS7OdWFrS62oV/K+vC3/CkFuUlg+l51o1F/E6uPgjxvhXIarGRGC+M41yzBPuajf3xGdwUl3nQSMFOH25FIVzVMRqe"
    "NZ3uGZiM4dWcx2UApdXeRY+n6lBxmusVxccc1shkkfLEnUDNCFJn09YaUIpTxklQxKJcP3KYKY4rw1tynLLY1+z0ki3r4pG9TM7V"
    "XqIiXlWNFdmfh+P2TEXhJUtFJSM5VjrDWxmHwp07YSGiuJLOiDmRWJqnJdJCVMgyZ9GcBpl4NVcm9I/UxwIaw1pWvOxrhBWOEaIj"
    "NjnamnzSzsXfZLrQMfIJsw/P/FOE2sQ/Y0+N5591PlWHJNti7sQ/9iBjOZ+mHU3R1znwmfIDuih/Erl24TKPHTLykadGz2kdZXJq"
    "vSDZnqtI6Qk6tRN294Kd0obGtBfXbQ17ZZvOY18yt7UQctKjtduAGZOC4y5alDEosibsOXMnelEKscNPFEmrgteRGI6H/JIVgqBT"
    "JZQWxNqDkXM+hbJIpDozrWSeELActdvBikG4bENrFwe0jm0E4mmsIt39j96Ht3cjwKdTaxV8GRM3R/lJKxvRyq/5W63/CxYmscCe"
    "ywuWyBInQRCiTdUC4YAv4B9immcsBCyLhOWI70iqyQKlYRAsbEQ7s9CMcAHrmXRP8+rObQI6qu5ycYWJEHkf8KyK1zX2oyRLwYvw"
    "loBbRK4GfoNrI+Zqex42mWqMye/Kq+Ox5xk5fdg4+qgVrhIpC9da5uluGDx9SlHv+byknbl6Z+3mewc+RI6iD4O5izVwTionlCCe"
    "qSMkVwZxRAfCiIrFsE7LOtU69qq2ZkLdiKQgmwX4DipZNeqwOz3DJ5VeVyK1Srgu5JXI8OfOBw04V7MX2o69uCQnm1ErOBvDGtBR"
    "+JFLmej1OGy5xhmabVJaH+kSY5+xQx0gwDhABNOftk6rwoROu8IcOheQGc6Co6k5/I2mBq94xb8peCrcz446qHOhMxCoDEVo4RPP"
    "QDQ845NwG0uhQV9oUQVB61Ylp6OzkJSoB/SY9EKXeSIpZB/J4AstDWdoG1yjpRWM0eoaz9DVwi1622f02ziKIbuJUWU9dBsrOp1J"
    "zQka+zoKetGpvtFWYIC2+r5v0KB4+HnF1xxooPMcTDGKiZHxRBA6JCyNy1Q2qMFEho6sBnNUCgh6buY3vlpkoRhdgK4lqkY4XwIX"
    "IvHRh50E73EwBDaKWSIyd+l53gLxmt+5np4+N35bQA+1/X0AuqVvrBXSlI7YpU6rlTze6iQOQhgAqRXKiTv2qeD6J7aOZqI2JE19"
    "W9RCA6Hep46NhV1UuukIzwYj8m1TTHDmFBjrUqKMVQ2dnCsYmbTLNnHW2WG3MPVWM7XJ7I0ikqa6f9iGpn7tGlGlv85gpayNe7uw"
    "mPzTyOPOb0ZqI4CQeI2CLQJ1/iuyoKcoeaDxxDXe9qC1FJUWA0uQuZbF1eKGLiC0qxu0buDIKNCxqs7XQQeCgGRBPcyOIFqJtGzE"
    "lCsKP5dxWcrMnfjnagKOOSGypZWrTBTZJm6164Pf+FaOWvTvr2+I713/kz7/9Y4+Z++I9GnFzss3pq4XOKUdJzSuy/i5WpcYZySD"
    "nK+3FqaJCnVgpU8QoSV4XsglPdOeVappqS5oGkxq835BSufqWtf4bcRUc9HXYUf1cP+Xep6uAzLtW6kPV5ah1jSA8EgdhbGV3bzj"
    "t1I2DVlXqNbvVbM15P+rSOFqjhDj1ETEUE8cJrKOrIUsmk0M2/5bmDJTl4LCcb5yPn9Vf4VgK6uin0VU0kbsxYvgxYsXyMt4xYoi"
    "AJECr/rKoGgzL/TuHQb8YeFICcUZ+4QYvaFQbet136YNv1k0Ia4GlbdoT/X9Xz5HgdFao4q2SmnjI/KXmdrQwN2yifwyg1Rn7PTq"
    "aAglGcUERWVABIOKmMKWCJyD8EXxNpKpRCJWMpPBRoD7xyFXpdEq5i1noHQgBHIQ3ps4C2NVZK1cA1FFgXoZw48q6x5+KyIkAj1/"
    "B2rH2TeCnr8V0PxJdb+twboefs60SKIz47Wof6wffsn05cws3/l+QlesCUXA+Lz9ONNBcN/RH1nGVbyRwwa0BLQRgaQ/1pQgNNI4"
    "8Z6uzalKtHUh63x552oy4mnA9lFqqSzaywGiCpi34LDnp2ADPNKAkadAUIOelrjf34zXvh/o66ziXJpC1po7hRMSUGaAcRyR3XRH"
    "LJXxRRkWuO0WsUKkii9xayODElDbur2tu+sxWbO5u37BDvHDaDbdPGq4XPOK2aBVDexfwF5tq41Cw9tu/Ibu9eJY5RSFreZ+dLXQ"
    "etLch+mzyD2iYA3phgACHOjKcWEVEZvFQZt6bMmeBm4a0O2G3+lKQv1dU2+tHyE6Lhr2pzrUk4m3ugoHi8fJJZvN7fP3Sgdmge7P"
    "/G5cM4Du2kSvlHBvQnmPKh+ItFKZ6xy8OzxIDw+itwd/vzj49uLgu387MI6taFieNfJOg5bbu75kG3TNfOOCNhpn89Yr/SOoY6pb"
    "PgFRwe9cgrmPFyP/aHXvEBVJOKLVGVEUfq9ohOnxqWdk93xPPb7jKXD8SMC4WXij7cVOdxdTaz0+63xr1p9o2nhw2o7m3T33JnnG"
    "mLlQpusCyVivubHK5Q8ipEvrh12rz/YzG1J7pZiKdagP+r0egfHLYo0cijO+O66G3/xtYGBKxKOZyBzDhIhS5uZNzcwuXR3bGeBH"
    "+q59y8HW81wU9E6Kl9qIM50lzdsERKRnu2ZA27TTxqPod3Rh28wxThuWMjY3HpppvrS76vFuzptjZzzTFaXOE+Xekqd5IlywrfAK"
    "M8mSWl0AhlipNybte3evvTjR9P3M2NDilh/jT8E0JtE1Z9an7rORUrfwenuZPLIPRQf2EHh1uuv/Bc5tKqyAsGu3YI+90r8OEGJa"
    "prl5y67iNoxUvRnLZJVx146Nm+4a44CcgOSCo4QCmsETFU1AkQVZAWnhgRaIlygNFA+vZUavKLBcVj262nZ90Q5vWARZ5XruXvs7"
    "Z/LjkhR47Cc2NMy+vrL+d2Ldn6/nA/ahWBhaYxDCzXUlovd8FAWdZ1qsvu6Hy/XoMddG0Y5Hu+vLHneSeW1ntu3W2jDivTd4tzTc"
    "FltsCG5rnDCZ24R1GHcoL8LE3Ken9P7N7+ZsmSx6fpA+P3hHEfGYWGtZs4eO3f6BLYxPH4E/rNoq28XAx6Qa3NLOpprcI40trGz2"
    "0c1o4+8MPB5R9N9GTPzsmBdN8ORM0aS1LO0AoIFr/bZO+U+9sjNvupoNUmBt7s3FuqqLTKt/8uTVP66vtxbpxyMaVJT0PaRQL7jM"
    "2DaRRLylEJVbMj1W2WzS6bHK5nFr5g63bHvoLTxcFNVhpcl1SzTNE66PYg3J0tlhnGhGsVQaqGObeKL18Kt6zY+eh1+bI/TZp21b"
    "iJYknTDcV5hCw4ffHIuZOqEsCiGbfw/YUvcj5RL6Gfbrli8622whBEpJnVCPMmEol8VesHCih185vZakjUB4ybNQ0uuWhKcPv6l/"
    "YBi7tHhinkCv6N2fAaIGXfSDUUBj9qGsmPy8PxJ6t2K9P1hUvZ3Hd8LNf1ycvZdjp4yZ6dBHgNjDLyTnbN2OrYa9z61c+fy4y7XS"
    "7Sy/aDqZbeFSMAQW7WlLgbpBXuhvOqY9fH//5L9whVkS"
)
exec(zlib.decompress(base64.b64decode(TEXTO)).decode())

datos = generar_datos()          # genera las tablas, con una semilla fija: todos los equipos de esta empresa ven lo mismo
globals().update(datos)          # cada tabla queda disponible como variable


In [ ]:
SLUG = "07_Vidriera"
import os, sqlite3
import pandas as pd

def inventario():
    """Tablas disponibles: filas, columnas y porcentaje de celdas faltantes."""
    return pd.DataFrame([(n, len(df), len(df.columns), round(float(df.isna().mean().mean() * 100), 1)) for n, df in datos.items()], columns=["tabla", "filas", "columnas", "% de faltantes"])

def diccionario(tabla=None):
    """Qué significa cada columna. Es una descripción neutra: no dice qué está limpio ni cuándo se conoce cada dato."""
    return pd.DataFrame([(t, c, d) for t, cols in DICC.items() if tabla in (None, t) for c, d in cols.items()], columns=["tabla", "columna", "qué es"])

bitacora_calidad = pd.DataFrame(columns=["problema", "dónde", "filas afectadas", "qué decidimos y por qué"])
def anotar(problema, donde, filas, decision):
    """Deja anotado un problema de calidad y la decisión tomada. Es parte de la trazabilidad."""
    bitacora_calidad.loc[len(bitacora_calidad)] = [problema, donde, filas, decision]; return bitacora_calidad

def exportar_superset(carpeta="superset", **tablas):
    """Guarda las tablas que pases como archivo SQLite y como CSV, para conectar Apache Superset."""
    os.makedirs(carpeta, exist_ok=True); con = sqlite3.connect(f"{carpeta}/{SLUG}.db")
    for nombre, df in tablas.items():
        df.to_csv(f"{carpeta}/{nombre}.csv", index=False); df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, con, index=False, if_exists="replace")
    con.close(); print(f"Listo: {carpeta}/{SLUG}.db y un CSV por tabla. Conectá Superset a ese archivo."); return carpeta


In [ ]:
inventario()

In [ ]:
diccionario()

### Tu definición del problema

Completá este diccionario. Si cambia más adelante, cambiá también la respuesta y anotá por qué.

In [ ]:
definicion = {
    "tipo de problema": "",
    "población": "",
    "período": "",
    "granularidad (una fila es…)": "",
    "variable objetivo": "",
    "qué se sabe en el momento de decidir": "",
}
definicion


## 2 · Python

**Qué tenés que resolver en tu caso:** Resolver pedidos duplicados por reintentos, direcciones y correos con variantes, importes con coma y con punto, y fechas en distintas zonas horarias.

Limpiá **sin borrar en silencio**: cada decisión se anota con `anotar(...)` y queda en la bitácora de calidad.

In [ ]:
# Tu trabajo


In [ ]:
bitacora_calidad

## 3 · Estadística

**Qué tenés que resolver en tu caso:** Tasa de fraude de alrededor del 0,8 % con su intervalo, y diferencias entre grupos de pedidos.

Mostrá distribuciones e intervalos, y al menos una comparación o prueba con su conclusión y su límite.

In [ ]:
# Tu trabajo


## 4 · Modelado

**Qué tenés que resolver en tu caso:** Regresión logística, Random Forest y XGBoost con pesos; validación temporal con una ventana de maduración; PR-AUC y recall en la cola de 40.

Empezá por una referencia simple. Validá sin mirar el futuro y elegí la métrica por el costo del error, no por costumbre.

In [ ]:
# Tu trabajo


## 5 · Deep Learning

**Qué tenés que resolver en tu caso:** Una red contra XGBoost con menos del 1 % de casos positivos: ¿se justifica?

Entrená una red y compará contra la mejor referencia con la misma validación. Decidí si vale su costo y su opacidad.

In [ ]:
# Tu trabajo


## 6 · Superset

**Qué tenés que resolver en tu caso:** Tablero con la cola de revisión, el fraude capturado y la fricción a clientes buenos.

Con `exportar_superset(nombre=tabla, ...)` llevás tus tablas a un archivo SQLite para conectar Apache Superset. Un gráfico por pregunta.

In [ ]:
# Tu trabajo


## 7 · Producción

**Qué tenés que resolver en tu caso:** Contrato de entrada, tiempo de respuesta en milisegundos, dos umbrales y deriva, porque quienes defraudan se adaptan.

Definí el contrato de entrada (campos, tipos y rangos), la versión, una prueba de humo y las señales de monitoreo con umbral, acción y responsable.

In [ ]:
# Tu trabajo


## 8 · Negocio

**Qué tenés que resolver en tu caso:** El costo de un contracargo frente al de frenar a un cliente bueno; explicación para el analista que revisa.

Decisión, costos, explicación global y local, y una recomendación proporcional a la evidencia.

In [ ]:
# Tu trabajo


## 9 · Fichas y recomendación

### Ficha de evidencia

| Campo | Tu respuesta |
| --- | --- |
| Decisión que se toma | |
| Población y período | |
| Procedimiento | |
| Resultado | |
| Validación | |
| Límite | |

### Ficha de modelo y plan de monitoreo

| Campo | Tu respuesta |
| --- | --- |
| Qué modelo se propone y por qué ese | |
| Con qué datos y qué versión | |
| Contrato de entrada | |
| Riesgos principales | |
| Señales a vigilar, umbral, acción y responsable | |

### Recomendación (una página)

- **Evidencia:** qué muestran los datos.
- **Interpretación:** qué creemos que significa, sin presentarlo como causalidad.
- **Recomendación:** qué conviene hacer, proporcional a la evidencia.
- **Una limitación y un siguiente paso.**